In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import datetime
from pyspark.sql import functions as F

In [0]:


trips_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("client_id", IntegerType(), True),
    StructField("driver_id", IntegerType(), True),
    StructField("city_id", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("request_date", DateType(), True)
])

trips_data = [
    (1, 1, 10, 1, "completed", datetime.strptime("2023-10-01", "%Y-%m-%d").date()),
    (2, 2, 11, 1, "completed", datetime.strptime("2023-10-01", "%Y-%m-%d").date()),
    (3, 3, 13, 2, "cancelled_by_client", datetime.strptime("2023-10-01", "%Y-%m-%d").date()),
    (4, 4, 14, 1, "cancelled_by_driver", datetime.strptime("2023-10-01", "%Y-%m-%d").date()),
    (5, 5, 10, 2, "completed", datetime.strptime("2023-10-01", "%Y-%m-%d").date()),
    (6, 1, 10, 1, "cancelled_by_client", datetime.strptime("2023-10-02", "%Y-%m-%d").date()),
    (7, 2, 11, 2, "completed", datetime.strptime("2023-10-02", "%Y-%m-%d").date()),
    (8, 3, 13, 1, "completed", datetime.strptime("2023-10-02", "%Y-%m-%d").date()),
    (9, 4, 14, 2, "cancelled_by_driver", datetime.strptime("2023-10-02", "%Y-%m-%d").date()),
    (10, 5, 11, 1, "completed", datetime.strptime("2023-10-02", "%Y-%m-%d").date()),
    (11, 1, 10, 1, "completed", datetime.strptime("2023-10-03", "%Y-%m-%d").date()),
    (12, 5, 11, 2, "cancelled_by_client", datetime.strptime("2023-10-03", "%Y-%m-%d").date()),
    (13, 3, 13, 1, "cancelled_by_driver", datetime.strptime("2023-10-03", "%Y-%m-%d").date()),
    (14, 2, 14, 2, "completed", datetime.strptime("2023-10-03", "%Y-%m-%d").date()),
    (15, 1, 10, 1, "completed", datetime.strptime("2023-10-04", "%Y-%m-%d").date())
]

trips_df = spark.createDataFrame(trips_data, schema=trips_schema)

# ==========================================
# 2. Users DataFrame
# ==========================================
users_schema = StructType([
    StructField("users_id", IntegerType(), True),
    StructField("banned", StringType(), True),
    StructField("role", StringType(), True)
])

users_data = [
    (1, "No", "client"),
    (2, "No", "client"),
    (3, "No", "client"),
    (4, "Yes", "client"),
    (5, "No", "client"),
    (10, "No", "driver"),
    (11, "No", "driver"),
    (13, "No", "driver")
]

users_df = spark.createDataFrame(users_data, schema=users_schema)

# Display DataFrames
print("Trips DataFrame:")
trips_df.show(truncate=False)

print("Users DataFrame:")
users_df.show()

In [0]:
trip_driver_df = trips_df.alias('t').join(users_df.alias('d'), (F.col("t.driver_id") == F.col("d.users_id")) & (F.col("d.banned") == "No"),'inner').join(users_df.alias("c"),(F.col("t.client_id") == F.col("c.users_id")) & (F.col("c.banned") == "No") ,'inner').filter(F.col("request_date").between("2023-10-01","2023-10-03"))


total_trips = trip_driver_df.groupBy(F.col('request_date').cast('date')).count()
cancel_trips = trip_driver_df.groupBy(F.col('request_date')).agg(F.sum(F.when(F.col("status") != "Completed", 1).otherwise(0)))

total_trips.show()
cancel_trips.show()

In [0]:
data = [
    ("alice","carrot",1) , ("bob","banana",3),
    ("alice","tomato",2) , ("bob","carrot",2),
    ("charlie","banana",5), ("bob","apple",1),
    ("alice","tomato",2), ("charlie","carrot",3)
]

df = spark.createDataFrame(data,['name','item','weight'])

item_totals = df.groupBy("name","item").agg(F.sum("weight").alias("weight"))

results = item_totals.groupBy("name")\
    .agg(
        F.collect_list(F.struct("item","weight")).alias("item_list"),
        F.sum("weight").alias("total_weight")
    )

results.show(truncate=False)

# Multi-File Sales Union

problem : The order processing system generates daily sales CSV files. Due to retry logic in the upstream pipeline, some orders appear in multiple daily files as exact duplicates. The finance team needs a single, consolidated dataset with no duplicate records and numeric amounts for theri reporting

Source files 
1. sales_day1.csv
2. sales_day2.csv
3. sales_day3.csv

In [0]:
from pyspark.sql import SparkSession,DataFrame
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
spark = SparkSession.builder.appName('ap_etl').getOrCreate()

In [0]:
sch = StructType([
    StructField('order_id', StringType()),
    StructField('product', StringType()),
    StructField('amount', DoubleType()),
    StructField('date', DateType())
])
sales_df = spark.read.format('csv').schema(sch).options(header=True).load("/Volumes/spark_data/dev/spark_data_volume/manish_etl/sales_day*.csv")

display(sales_df)

In [0]:
sch = StructType([
    StructField('order_id', StringType()),
    StructField('product', StringType()),
    StructField('amount', DoubleType()),
    StructField('date', DateType())
])


csv_path = "/Volumes/spark_data/dev/spark_data_volume/manish_etl/sales/"

In [0]:
def extract(spark, input_path: str) -> DataFrame:
    """Read all CSV files from the input directory."""
    df = spark.read.format('csv').option('header','true').load(input_path)
    return df


df = extract(spark,csv_path)

df.show()





In [0]:
from pyspark.sql import functions as F
def transform(df: DataFrame) -> DataFrame:

    """Remove duplicates and cast amount to double."""
    df = df.dropDuplicates().withColumn("amount", F.col("amount").cast("double")).orderBy("order_id")
    return df

silver_df = transform(df)   
silver_df.show() 

In [0]:
def load(df: DataFrame, output_path: str):
    """Write the deduplicated DataFrame as Parquet to output_path."""
    df.write.format('parquet').mode('overwrite').save(output_path)


output_path = "/Volumes/spark_data/dev/spark_data_volume/manish_etl/sales_part"

load(silver_df,output_path)





In [0]:
df = spark.read.parquet("/Volumes/spark_data/dev/spark_data_volume/manish_etl/sales_part/")
df.show()